# Phase 8: Structural Root-Cause Analysis of Persistent Ranking Failures

## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from pathlib import Path
from itertools import product
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

warnings.filterwarnings('default')
np.random.seed(36)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths
PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT / 'data' / 'raw'
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models'
PHASE8_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase8_structural_analysis'
PHASE8_OUTPUT.mkdir(parents=True, exist_ok=True)

#Config space
MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']
DATASET='2007'
BASELINE_KEY='pointwise_raw_2007'  

#Statistical constants
FDR_ALPHA=0.05
N_BOOTSTRAP=2000
EFFECT_THRESHOLD_CONTINUOUS=0.10
EFFECT_THRESHOLD_CATEGORICAL=0.01
MIN_SAMPLE_WARNING=10

print('='*80)
print('PHASE 8: STRUCTURAL ROOT-CAUSE ANALYSIS')
print('='*80)
print(f'Output directory:{PHASE8_OUTPUT}')
print(f'Baseline config:{BASELINE_KEY}')
print(f'Dataset focus:MQ{DATASET}')
print(f'FDR alpha:{FDR_ALPHA}')
print(f'Effect thresh (cont):{EFFECT_THRESHOLD_CONTINUOUS}')
print(f'Effect thresh (cat):{EFFECT_THRESHOLD_CATEGORICAL}')
print(f'Bootstrap reps:{N_BOOTSTRAP}')
print(f'Min sample warning :{MIN_SAMPLE_WARNING}')
print('='*80)

PHASE 8: STRUCTURAL ROOT-CAUSE ANALYSIS
Output directory:b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase8_structural_analysis
Baseline config:pointwise_raw_2007
Dataset focus:MQ2007
FDR alpha:0.05
Effect thresh (cont):0.1
Effect thresh (cat):0.01
Bootstrap reps:2000
Min sample warning :10


## 2. Utility functions

In [8]:

# 1.failure-flag parser 

def make_fail_flag(series: pd.Series) -> pd.Series:
    #Convert Failure@5_primary to clean 0/1 int. Unknown -> 0 (conservative).
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float).round().astype(int).clip(0, 1)
    _T = {'true', '1', 'yes', '1.0'}
    _F = {'false', '0', 'no', '0.0', ''}
    def _p(v):
        if pd.isna(v): return 0
        if isinstance(v, bool): return int(v)
        if isinstance(v, (int, float)):
            return 0 if np.isnan(float(v)) else int(round(float(v)))
        if isinstance(v, str):
            vl=v.strip().lower()
            if vl in _T: return 1
            if vl in _F: return 0
        return 0
    return series.map(_p).astype(int)

#Self-test
_test = pd.Series([True, False, 'True', 'false', '1', '0', 1.0, 0.0, pd.NA, None])
assert list(make_fail_flag(_test)) == [1, 0, 1, 0, 1, 0, 1, 0, 0, 0], 'make_fail_flag FAILED self-test'
del _test
print('make_fail_flag')


# 2. Cliff's delta 
def cliffs_delta_unpaired(a: np.ndarray, b: np.ndarray) -> float:
    
    a=np.asarray(a, dtype=float)
    b=np.asarray(b, dtype=float)

    #Dropping NaNs (keeps stats stable if caller forgets)
    a=a[~np.isnan(a)]
    b=b[~np.isnan(b)]

    n_a, n_b=len(a), len(b)
    if n_a==0 or n_b==0:
        return 0.0

    a_sorted=np.sort(a)
    b_sorted=np.sort(b)

    #Counting pairs a > b and a < b using two-pointer scans
    greater=0
    j=0
    for i in range(n_a):
        while j < n_b and b_sorted[j] < a_sorted[i]:
            j += 1
        greater += j  # number of b values < a[i]

    less=0
    j=0
    for i in range(n_a):
        while j < n_b and b_sorted[j] <= a_sorted[i]:
            j += 1
        less += (n_b - j)  # number of b values > a[i]

    return float((greater - less) / (n_a * n_b))

print('cliffs_delta_unpaired')



# 3. Bootstrap CI for median difference 

def bootstrap_median_diff(a: np.ndarray, b: np.ndarray, n_boot: int = N_BOOTSTRAP) -> tuple:
    #Bootstrap 95% CI for median(a) - median(b).
    if len(a)==0 or len(b)==0:
        return 0.0, 0.0
    diffs=[]
    for _ in range(n_boot):
        a_sample=a[np.random.randint(0, len(a), len(a))]
        b_sample=b[np.random.randint(0, len(b), len(b))]
        diffs.append(np.median(a_sample) - np.median(b_sample))
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

print('bootstrap_median_diff')



# 4. Bootstrap CI for risk difference 

def bootstrap_risk_diff(a_binary: np.ndarray, b_binary: np.ndarray, n_boot: int = N_BOOTSTRAP) -> tuple:
    #Bootstrap 95% CI for risk_diff=mean(a) - mean(b).
    if len(a_binary)==0 or len(b_binary)==0:
        return 0.0, 0.0
    diffs=[]
    for _ in range(n_boot):
        a_sample=a_binary[np.random.randint(0, len(a_binary), len(a_binary))]
        b_sample=b_binary[np.random.randint(0, len(b_binary), len(b_binary))]
        diffs.append(a_sample.mean()-b_sample.mean())
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

print('bootstrap_risk_diff')



# 5. BH-FDR correction

def bh_fdr(pvals: list, alpha: float = FDR_ALPHA) -> np.ndarray:
    #Benjamini-Hochberg FDR correction.
    pvals=np.asarray(pvals, dtype=float)
    n=len(pvals)
    if n==0:
        return np.array([])
    order=np.argsort(pvals)
    ranks=np.arange(1, n + 1)
    adj=np.minimum(pvals[order] * n / ranks, 1.0)
    adj=np.minimum.accumulate(adj[::-1])[::-1]
    q=np.empty(n)
    q[order]=adj
    return q

print('bh_fdr')



# 6. Alignment safety check 

def check_alignment_or_raise(test_df: pd.DataFrame, pred_df: pd.DataFrame) -> tuple:
    """
    Verify alignment keys exist between test features and predictions.
    Returns (alignment_ok: bool, keys: list).
    """
    #Checking (qid, docid)
    if 'docid' in test_df.columns and 'docid' in pred_df.columns:
        print('Alignment: (qid, docid) present in both')
        #Quick validation
        test_pairs=set(zip(test_df['qid'], test_df['docid']))
        pred_pairs=set(zip(pred_df['qid'], pred_df['docid']))
        if len(test_pairs & pred_pairs) > 0:
            print(f'{len(test_pairs & pred_pairs)} matching (qid,docid) pairs found')
            return True, ['qid', 'docid']
        else:
            print('No matching (qid,docid) pairs -> UNSAFE')
            return False, []
    
    #Checking (qid, doc_idx) with validation
    if 'doc_idx' in test_df.columns and 'doc_idx' in pred_df.columns:
        print('Found (qid, doc_idx) - validating alignment...')
        #Checking per-qid doc counts match
        test_counts=test_df.groupby('qid').size()
        pred_counts=pred_df.groupby('qid').size()
        common_qids=set(test_counts.index) & set(pred_counts.index)
        if len(common_qids)==0:
            print('No common qids -> UNSAFE')
            return False, []
        mismatches=sum(test_counts[q] != pred_counts[q] for q in common_qids if q in test_counts.index and q in pred_counts.index)
        if mismatches > 0:
            print(f'{mismatches} qids have different doc counts -> UNSAFE')
            return False, []
        #Checking if doc_idx are contiguous 0..n-1 per qid
        for qid in list(common_qids)[:5]:  #spot check
            test_idx=sorted(test_df[test_df['qid']==qid]['doc_idx'].values)
            pred_idx=sorted(pred_df[pred_df['qid']==qid]['doc_idx'].values)
            if test_idx != pred_idx or test_idx != list(range(len(test_idx))):
                print(f'doc_idx not contiguous or mismatched for qid={qid} -> UNSAFE')
                return False, []
        print('doc_idx alignment validated')
        return True, ['qid', 'doc_idx']
    
    #No reliable alignment
    print('NO RELIABLE ALIGNMENT KEYS (no docid, no valid doc_idx)')
    print('-> Feature <-> score association SKIPPED')
    return False, []

print('check_alignment_or_raise')

print('\nAll utility functions defined with guards')

make_fail_flag
cliffs_delta_unpaired
bootstrap_median_diff
bootstrap_risk_diff
bh_fdr
check_alignment_or_raise

All utility functions defined with guards


## 3. Loading Phase 6 artifacts

In [9]:
print('='*80)
print('LOADING PHASE 6 ARTIFACTS (MQ2007) WITH VALIDATION')
print('='*80)

all_query_metrics={}
all_predictions={}

for model, pipeline in product(MODELS, PIPELINES):
    key=f'{model}_{pipeline}_{DATASET}'
    
    qm_file=PHASE6_OUTPUT / f'{key}_query_metrics.csv'
    pred_file=PHASE6_OUTPUT / f'{key}_predictions.csv'
    
    #Hard fail if missing
    if not qm_file.exists():
        raise RuntimeError(f'MISSING FILE: {qm_file}')
    if not pred_file.exists():
        raise RuntimeError(f'MISSING FILE: {pred_file}')
    
    qm=pd.read_csv(qm_file)
    pred=pd.read_csv(pred_file)
    
    #Validating required columns
    required_qm=['qid', 'num_docs', 'num_relevant_1', 'Failure@5_primary']
    for col in required_qm:
        if col not in qm.columns:
            raise RuntimeError(f'MISSING COLUMN "{col}" in {qm_file}')
    
    required_pred=['qid', 'label', 'score']
    for col in required_pred:
        if col not in pred.columns:
            raise RuntimeError(f'MISSING COLUMN "{col}" in {pred_file}')
    
    #Parsing failure flag
    qm['_fail_flag']=make_fail_flag(qm['Failure@5_primary'])
    
    all_query_metrics[key]=qm
    all_predictions[key]=pred
    print(f'{key:35s} ({len(qm)} queries, {len(pred)} docs)')

expected=len(MODELS) * len(PIPELINES)
assert len(all_query_metrics)==expected, f'Expected {expected} configs, got {len(all_query_metrics)}'

print(f'\nLoaded {len(all_query_metrics)} / {expected} configs')
print('='*80)

LOADING PHASE 6 ARTIFACTS (MQ2007) WITH VALIDATION
pointwise_raw_2007                  (336 queries, 13652 docs)
pointwise_global_2007               (336 queries, 13652 docs)
pointwise_per_query_2007            (336 queries, 13652 docs)
pairwise_raw_2007                   (336 queries, 13652 docs)
pairwise_global_2007                (336 queries, 13652 docs)
pairwise_per_query_2007             (336 queries, 13652 docs)
lightgbm_raw_2007                   (336 queries, 13652 docs)
lightgbm_global_2007                (336 queries, 13652 docs)
lightgbm_per_query_2007             (336 queries, 13652 docs)

Loaded 9 / 9 configs


## 4. Setting Baseline Reference Config

In [10]:
reference_qm=all_query_metrics[BASELINE_KEY]
reference_pred=all_predictions[BASELINE_KEY]

print(f'Baseline reference config:{BASELINE_KEY}')
print(f'Query metrics:{len(reference_qm)} queries')
print(f'Predictions:{len(reference_pred)} documents')

Baseline reference config:pointwise_raw_2007
Query metrics:336 queries
Predictions:13652 documents


## 5. Building Query Groups (Evaluable + Persistent)

In [11]:
print('\n'+'='*80)
print('BUILDING QUERY GROUPS')
print('='*80)

failure_sets={}
evaluable_sets={}

for key, qm in all_query_metrics.items():
    evaluable_sets[key]=set(qm.loc[qm['num_relevant_1'] > 0, 'qid'].values)
    fail_mask=(qm['num_relevant_1'] > 0) & (qm['_fail_flag']==1)
    failure_sets[key]=set(qm.loc[fail_mask, 'qid'].values)
    parts=key.split('_')
    print(f'{parts[0]:10s} {parts[1]:10s}: {fail_mask.sum():4d} failures')

#Building groups
all_evaluable_qids=set.intersection(*evaluable_sets.values())
persistent_qids=set.intersection(*failure_sets.values())
all_failing_qids=set.union(*failure_sets.values())
non_persistent_qids=all_failing_qids-persistent_qids
successful_qids=all_evaluable_qids-all_failing_qids

#Computing BOTH persistent percentages 
pct_of_failing=100*len(persistent_qids) / len(all_failing_qids) if len(all_failing_qids) > 0 else 0.0
pct_of_evaluable=100 * len(persistent_qids) / len(all_evaluable_qids) if len(all_evaluable_qids) > 0 else 0.0

print('\n'+'='*80)
print('QUERY GROUP SUMMARY')
print('='*80)
print(f'Evaluable (intersection):{len(all_evaluable_qids)}')
print(f'Persistent (fail in all 9 configs):{len(persistent_qids)}')
print(f'Non-persistent (fail in ≥1, <9):{len(non_persistent_qids)}')
print(f'Successful (never fail):{len(successful_qids)}')
print(f'\n% Persistent of failing queries:{pct_of_failing:.1f}%')
print(f'% Persistent of evaluable queries:{pct_of_evaluable:.1f}%')
print('='*80)

#Sample size warnings
warnings_issued=[]
for name, qids in [('Persistent', persistent_qids), ('Non-persistent', non_persistent_qids), ('Successful', successful_qids)]:
    if len(qids)<MIN_SAMPLE_WARNING:
        msg = f'{name} group has {len(qids)} queries (< {MIN_SAMPLE_WARNING}) -> unstable inference'
        print(f'\nWARNING: {msg}')
        warnings_issued.append(msg)


BUILDING QUERY GROUPS
pointwise  raw       :   44 failures
pointwise  global    :   46 failures
pointwise  per       :   56 failures
pairwise   raw       :   50 failures
pairwise   global    :   50 failures
pairwise   per       :   51 failures
lightgbm   raw       :   51 failures
lightgbm   global    :   53 failures
lightgbm   per       :   44 failures

QUERY GROUP SUMMARY
Evaluable (intersection):290
Persistent (fail in all 9 configs):22
Non-persistent (fail in ≥1, <9):63
Successful (never fail):205

% Persistent of failing queries:25.9%
% Persistent of evaluable queries:7.6%


After constructing query groups across all 9 model–pipeline configurations (3 models × 3 normalization strategies), we identified 290 evaluable queries (i.e., queries with at least one relevant document across all configurations). Among these, 85 (22+63) queries fail in at least one configuration. Of those failing queries, 22 are persistent failures, meaning they fail in all 9 configurations, while 63 are non-persistent failures that fail in some but not all configurations. The remaining 205 queries are consistently successful and never fail in any configuration. Persistent failures represent 25.9% of all failing queries and 7.6% of the total evaluable query set. This indicates that while most queries are handled successfully by at least one configuration, a meaningful subset of failures (approximately one quarter of all failing queries) appears structurally robust to model and normalization changes, motivating deeper structural root-cause analysis in the next steps.